In [1]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedShuffleSplit
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
data = pd.read_csv("Cleaned_data.csv")

In [3]:
df=pd.DataFrame(data)

In [4]:
split=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
for train_index , test_index in split.split(df,df["Churn"]):
    train_data=df.loc[train_index]
    test_data=df.loc[test_index]

In [5]:
train_data_label = train_data["Churn"]
train_data=train_data.drop("Churn",axis=1)
test_data_label = test_data["Churn"]
test_data=test_data.drop("Churn",axis=1)

In [6]:
train_data_label.info()

<class 'pandas.core.series.Series'>
Index: 79592 entries, 38383 to 88472
Series name: Churn
Non-Null Count  Dtype  
--------------  -----  
79592 non-null  float64
dtypes: float64(1)
memory usage: 1.2 MB


In [7]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 79592 entries, 38383 to 88472
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Gender                79592 non-null  object 
 1   Age                   77892 non-null  float64
 2   City                  79592 non-null  object 
 3   State                 79592 non-null  object 
 4   AccountType           79592 non-null  object 
 5   Branch                79592 non-null  object 
 6   DateOfJoining         71825 non-null  object 
 7   Occupation            79592 non-null  object 
 8   CreditScore           76401 non-null  float64
 9   TenureYears           79506 non-null  float64
 10  AccountBalance_INR    77116 non-null  float64
 11  MonthlySalary_INR     75629 non-null  float64
 12  NumProducts           79592 non-null  int64  
 13  LoanAmount_INR        79592 non-null  int64  
 14  FD_Amount_INR         79592 non-null  int64  
 15  HasCreditCard       

In [8]:
train_data = train_data.drop("DateOfJoining",axis=1)

In [9]:
test_data = test_data.drop("DateOfJoining",axis=1)

In [10]:
cat_col = train_data.select_dtypes(include="object").columns.tolist()
num_col = train_data.select_dtypes(include=["int64","float64"]).columns.tolist()

In [11]:
num_pipeline = Pipeline([
    ("impute",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("encode",OneHotEncoder(handle_unknown = "ignore"))
])

general_pipeline = ColumnTransformer([
    ("num",num_pipeline,num_col),
    ("cat",cat_pipeline,cat_col)
])

In [12]:
trained_prepared_data = general_pipeline.fit_transform(train_data)

In [13]:
trained_prepared_data.shape

(79592, 80)

In [14]:
test_prepared_data = general_pipeline.transform(test_data)

In [15]:
model1 = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)
model1.fit(trained_prepared_data,train_data_label)
pred_model1 = model1.predict(test_prepared_data)
from sklearn.metrics import accuracy_score
accuracy_model1 = accuracy_score(test_data_label,pred_model1)
print(accuracy_model1)

0.6460123624302728


In [16]:
model2 = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)
model2.fit(trained_prepared_data,train_data_label)
pred_model2 = model2.predict(test_prepared_data)
accuracy_model2 = accuracy_score(test_data_label,pred_model2)
print(accuracy_model2)

0.821096537514448


In [17]:
model3 = XGBClassifier(n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42)
model3.fit(trained_prepared_data,train_data_label)
pred_model3 = model3.predict(test_prepared_data)
accuracy_model3 = accuracy_score(test_data_label,pred_model3)
print(accuracy_model3)

0.8181818181818182


In [18]:
from sklearn.metrics import classification_report

print(classification_report(test_data_label, pred_model1))
print(classification_report(test_data_label, pred_model2))
print(classification_report(test_data_label, pred_model3))

              precision    recall  f1-score   support

         0.0       0.89      0.64      0.75     16305
         1.0       0.29      0.65      0.40      3594

    accuracy                           0.65     19899
   macro avg       0.59      0.65      0.57     19899
weighted avg       0.78      0.65      0.69     19899

              precision    recall  f1-score   support

         0.0       0.82      1.00      0.90     16305
         1.0       0.76      0.01      0.03      3594

    accuracy                           0.82     19899
   macro avg       0.79      0.51      0.46     19899
weighted avg       0.81      0.82      0.74     19899

              precision    recall  f1-score   support

         0.0       0.83      0.99      0.90     16305
         1.0       0.47      0.05      0.09      3594

    accuracy                           0.82     19899
   macro avg       0.65      0.52      0.49     19899
weighted avg       0.76      0.82      0.75     19899



In [19]:
import pickle

In [22]:
import pickle

with open("preprocessing.pkl","wb") as f:
    pickle.dump(general_pipeline,f)

with open("logistic_model.pkl","wb") as f:
    pickle.dump(model1,f)

In [23]:
train_data["Occupation"].unique()

array(['Student', 'Salaried', 'Self-Employed', 'Retired', 'Business',
       'Homemaker', 'Unknown', 'Freelancer', 'Government Employee'],
      dtype=object)